# Bölüm 7: Sohbet Uygulamaları Oluşturma
## Microsoft Foundry Modelleri API Hızlı Başlangıç

Bu not defteri, [Azure OpenAI Örnekleri Deposu](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) temel alınarak hazırlanmıştır ve [Azure OpenAI](notebook-azure-openai.ipynb) hizmetlerine erişen not defterlerini içermektedir.

> **Not:** GitHub Modelleri Temmuz 2026 sonunda kullanımdan kaldırılacaktır. Bu not defteri artık aynı denemesi ücretsiz model kataloğu ve Azure AI Inference SDK deneyimi sunan [Microsoft Foundry Modelleri](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) hedeflemektedir.


# Genel Bakış  
"Büyük dil modelleri, metni metne eşleyen fonksiyonlardır. Verilen bir giriş metin dizisi için, büyük bir dil modeli, ardından gelecek metni tahmin etmeye çalışır"(1). Bu "hızlı başlangıç" defteri, kullanıcılara yüksek seviyede LLM kavramlarını, AML ile başlamada gerekli çekirdek paket gereksinimlerini, teşvik tasarımına yumuşak bir giriş ve çeşitli kullanım durumlarına dair birkaç kısa örneği tanıtacaktır. 


## İçindekiler  

[Genel Bakış](#overview)  
[OpenAI Hizmetini Nasıl Kullanılır](#how-to-use-openai-service)  
[1. OpenAI Hizmetinizi Oluşturma](#1.-creating-your-openai-service)  
[2. Kurulum](#2.-installation)    
[3. Kimlik Bilgileri](#3.-credentials)  

[Kullanım Alanları](#use-cases)    
[1. Metni Özetleme](#1.-summarize-text)  
[2. Metni Sınıflandırma](#2.-classify-text)  
[3. Yeni Ürün İsimleri Oluşturma](#3.-generate-new-product-names)  
[4. Bir Sınıflandırıcıyı İnce Ayar Yapma](#4.fine-tune-a-classifier)  

[Referanslar](#references)


### İlk isteminizi oluşturun  
Bu kısa egzersiz, Microsoft Foundry Modellerinde basit bir görev olan "özetleme" için modellere istem gönderme konusunda temel bir giriş sağlayacaktır.  


**Adımlar**:  
1. Python ortamınıza `azure-ai-inference` kütüphanesini kurun, henüz yapmadıysanız.  
2. Standart yardımcı kütüphaneleri yükleyin ve Microsoft Foundry Modelleri kimlik bilgilerinizi ayarlayın.  
3. Göreviniz için bir model seçin  
4. Model için basit bir istem oluşturun  
5. İsteğinizi model API'sine gönderin!  


### 1. `azure-ai-inference` Yükleyin


In [ ]:
%pip install azure-ai-inference

### 2. Yardımcı kitaplıkları içe aktarın ve kimlik bilgilerini oluşturun


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Doğru modeli bulmak  
GPT-4o ve GPT-4o mini gibi modeller doğal dili anlayabilir ve üretebilir, ayrıca Meta, Mistral, Cohere ve Microsoft modelleriyle birlikte Microsoft Foundry Modelleri kataloğunda mevcuttur.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-4o-mini"


## 4. Prompt Design  

"The magic of large language models is that by being trained to minimize this prediction error over vast quantities of text, the models end up learning concepts useful for these predictions. For example, they learn concepts like"(1):

* how to spell
* how grammar works
* how to paraphrase
* how to answer questions
* how to hold a conversation
* how to write in many languages
* how to code
* etc.

#### How to control a large language model  
"Of all the inputs to a large language model, by far the most influential is the text prompt(1).

Large language models can be prompted to produce output in a few ways:

Instruction: Tell the model what you want
Completion: Induce the model to complete the beginning of what you want
Demonstration: Show the model what you want, with either:
A few examples in the prompt
Many hundreds or thousands of examples in a fine-tuning training dataset"



#### There are three basic guidelines to creating prompts:

**Show and tell**. Make it clear what you want either through instructions, examples, or a combination of the two. If you want the model to rank a list of items in alphabetical order or to classify a paragraph by sentiment, show it that's what you want.

**Provide quality data**. If you're trying to build a classifier or get the model to follow a pattern, make sure that there are enough examples. Be sure to proofread your examples — the model is usually smart enough to see through basic spelling mistakes and give you a response, but it also might assume this is intentional and it can affect the response.

**Check your settings.** The temperature and top_p settings control how deterministic the model is in generating a response. If you're asking it for a response where there's only one right answer, then you'd want to set these lower. If you're looking for more diverse responses, then you might want to set them higher. The number one mistake people make with these settings is assuming that they're "cleverness" or "creativity" controls.


Source: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. Gönder!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Aynı çağrıyı tekrarlayın, sonuçlar nasıl karşılaştırılır?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Metni Özetle  
#### Zorluk  
Bir metin pasajının sonuna 'tl;dr:' ekleyerek metni özetleyin. Modelin herhangi bir ek talimat olmadan nasıl birçok görevi anladığını ve gerçekleştirdiğini fark edin. Modelin davranışını değiştirmek ve aldığınız özetlemeyi özelleştirmek için tl;dr'den daha açıklayıcı istemlerle deney yapabilirsiniz(3).  

Son çalışmalar, büyük bir metin korpusunda ön eğitim yapıldıktan sonra belirli bir görevde ince ayar yaparak birçok NLP görevi ve kıyaslamasında önemli ilerlemeler sağlandığını göstermiştir. Genellikle görevden bağımsız bir mimariye sahip olsa da, bu yöntem hâlâ binlerce veya onbinlerce örnekten oluşan görev özelinde ince ayar veri setleri gerektirir. Buna karşın, insanlar genellikle sadece birkaç örnekten veya basit talimatlardan yeni bir dil görevini gerçekleştirebilir - bu, mevcut NLP sistemlerinin hâlâ büyük ölçüde zorlandığı bir şeydir. Burada, dil modellerinin ölçeklendirilmesinin görevden bağımsız, az örnekle performansı büyük ölçüde iyileştirdiğini ve bazen önceki en iyi ince ayar yaklaşımlarıyla rekabet edebilir hale geldiğini gösteriyoruz.  



Özetle  


# Birkaç kullanım durumu için egzersizler  
1. Metni Özetle  
2. Metni Sınıflandır  
3. Yeni Ürün İsimleri Oluştur


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Metni Sınıflandır  
#### Görev  
Öğrenme anında verilen kategorilere öğeleri sınıflandırın. Aşağıdaki örnekte, hem kategorileri hem de sınıflandırılacak metni istemde sağlıyoruz (*playground_reference).  

Müşteri Talebi: Merhaba, dizüstü bilgisayarımın klavyesindeki tuşlardan biri yakın zamanda kırıldı ve bir yedeğe ihtiyacım olacak:  

Sınıflandırılmış kategori:  


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Yeni Ürün İsimleri Üret
#### Zorluk
Örnek kelimelerden ürün isimleri oluşturun. Burada, isimlerini üreteceğimiz ürün hakkında bilgiyi isteme dahil ediyoruz. Ayrıca almak istediğimiz deseni göstermek için benzer bir örnek veriyoruz. Ayrıca daha rastgele ve yenilikçi yanıtlar almak için sıcaklık değerini yüksek ayarladık.

Ürün açıklaması: Ev tipi milkshake yapıcı
Tohum kelimeler: hızlı, sağlıklı, kompakt.
Ürün isimleri: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Ürün açıklaması: Her ayak numarasına uyabilecek bir çift ayakkabı.
Tohum kelimeler: uyarlanabilir, uyumlu, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Referanslar  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portalı](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Metin sınıflandırmak için GPT modellerini ince ayar yapmanın en iyi uygulamaları](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Daha Fazla Yardım İçin  
[OpenAI Ticarileştirme Ekibi](AzureOpenAITeam@microsoft.com) 


# Katkıda Bulunanlar
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
